**Step 1: Install Required Libraries**

 In the first cell we are installing the necessary libraries

 1)spacy: for natural language processing

 2)pdfminer.six: to extract text from PDF resumes

 3)scikit-learn: for TF-IDF and cosine similarity

 4)pandas: for handling data tables

 #I did not use the streamlit library as it is not supported by google collab.

In [ ]:
!pip install -q spacy pdfminer.six scikit-learn pandas
!python -m spacy download en_core_web_sm

 **Step 2: Upload Resumes and Job Description**
         
         


   Let's user upload-     
  *   One or more PDF resumes
  *   One text file containing the job description

In [ ]:
from google.colab import files

uploaded_resumes = files.upload()
job_description_file = files.upload()



**Step 3: Parse Text from PDF Resumes**

A function pdfminer is defined to extract plain text from a pdf file.



In [ ]:
from pdfminer.high_level import extract_text

def extract_resume_text(pdf_path):
    try:
        return extract_text(pdf_path)
    except Exception as e:
        print(f"Error reading {pdf_path}: {e}")
        return ""

**Step 4: Preprocess Text Using spaCy**

   Cleans and preprocesses text:

        

*  Lowercases
  
*  Removes stopwords and punctuation



*   Lemmatizes (gets base word forms)

In [ ]:
import spacy
import re

nlp = spacy.load("en_core_web_sm")

def preprocess_text(text):
    text = re.sub(r'\s+', ' ', text)  # Remove excess whitespace
    doc = nlp(text.lower())
    tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]
    return ' '.join(tokens)

**Step 5: Convert to TF-IDF and Calculate Similarity**

 Reads and preprocesses the job description


*   Preprocesses each resume.
*   Converts texts to TF-IDF vectors.

*   Computes cosine similarity between the job description and each resume.
*   Sorts candidates by similarity score into a pandas DataFrame.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# Extract and preprocess job description
jd_filename = list(job_description_file.keys())[0]
# Use the extract_resume_text function to read the PDF job description
job_description_raw = extract_resume_text(jd_filename)
job_description_clean = preprocess_text(job_description_raw)

# Parse and preprocess resumes
resume_texts = []
resume_names = []

for filename in uploaded_resumes.keys():
    text = extract_resume_text(filename)
    clean_text = preprocess_text(text)
    resume_texts.append(clean_text)
    resume_names.append(filename)

# TF-IDF Vectorization
vectorizer = TfidfVectorizer()
vectors = vectorizer.fit_transform([job_description_clean] + resume_texts)

# Compute cosine similarity (JD is vector[0])
similarities = cosine_similarity(vectors[0:1], vectors[1:]).flatten()

# Create DataFrame
results_df = pd.DataFrame({
    'Candidate': resume_names,
    'Similarity Score': similarities
}).sort_values(by='Similarity Score', ascending=False)

results_df.reset_index(drop=True, inplace=True)
results_df

**Step 6: Export Results to CSV**

Exports the ranked candidates as a CSV file and lets you download it to your computer.



In [ ]:
results_df.to_csv("ranked_candidates.csv", index=False)
files.download("ranked_candidates.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>